# Figpack: bipolar LFP with seizures and spikes

Run `0_Seizure_detection.ipynb` first to save event tables

In [ ]:
from pathlib import Path
import pandas as pd
import spyglass.lfp as lfp
from spyglass.lfp.analysis.v1 import lfp_band
from seizure_detection_ll.utils import bipolar_montage, probe_bad_channels, create_lfp_seizure_spike_figurl


In [ ]:
rat_name, date, epoch_to_plot = "SC137", "20260218", "11_s6"
nwb_file_name = f"{rat_name}{date}_.nwb"
lfp_electrode_group_name = f"all_hippocampal_probes_all128ch_{rat_name}"
filter_name = "IED filt 1-250hz fs 1khz"
lfp_band_sampling_rate = 1000
merge_id = (lfp.LFPOutput.LFPV1() & {"nwb_file_name": nwb_file_name,
            "lfp_electrode_group_name": lfp_electrode_group_name,
            "target_interval_list_name": epoch_to_plot}).fetch1("merge_id")

# Create LFP band selection and populate it first if needed:
# lfp_band.LFPBandSelection().set_lfp_band_electrodes(
#     nwb_file_name=nwb_file_name, lfp_merge_id=merge_id,
#     electrode_list=list(range(256, 640)), filter_name=filter_name,
#     interval_list_name=epoch_to_plot, reference_electrode_list=[-1],
#     lfp_band_sampling_rate=lfp_band_sampling_rate)
# lfp_band.LFPBandV1().populate(lfp_band.LFPBandSelection() & {
#     "lfp_merge_id": merge_id, "filter_name": filter_name})

lfp_band_key = (lfp_band.LFPBandSelection() & dict(lfp_merge_id=merge_id,
                filter_name=filter_name, lfp_band_sampling_rate=lfp_band_sampling_rate)).fetch1("KEY")
filtered_lfp = (lfp_band.LFPBandV1() & lfp_band_key).fetch1_dataframe()
bad_channels = probe_bad_channels(nwb_file_name)


In [ ]:
out_dir = Path("/home/sc4712/src/repos/seizure_detection_ll/results") / f"{rat_name}{date}"
seizures = pd.read_csv(out_dir / "seizures.csv").query("epoch == @epoch_to_plot")
spikes = pd.read_csv(out_dir / "spikes.csv").query("epoch == @epoch_to_plot")

probe_figurls = {}
for probe_idx, name in enumerate(("L_hpc1", "L_hpc2", "R_hpc")):
    lfp_probe = filtered_lfp.iloc[:, probe_idx*128:(probe_idx+1)*128]
    bipolar_lfp = bipolar_montage(lfp_probe, bad_channels[probe_idx])
    label = f"{rat_name}{date}_{epoch_to_plot}_{name}_1khz_250hz"
    probe_figurls[name] = create_lfp_seizure_spike_figurl(
        bipolar_lfp, seizures, spikes, downsample=20, url_label=label)
    print(name, probe_figurls[name])
